In [ ]:
"""
Sampling dan Quantization untuk Pengenalan Suara
=================================================
Implementasi berdasarkan BAB III - Metode Penelitian
Referensi: "Perbandingan Kinerja Algoritma GMM dan TDNN
            dalam Pengenalan Suara Bahasa Jawa"

Alur:
  1. Load audio (FLAC/WAV)
  2. Konversi FLAC -> WAV (jika diperlukan)
  3. Sampling  -> nilai amplitudo kontinu per frame
  4. Quantization (16-bit PCM) -> nilai amplitudo diskrit
"""

# def convert_flac_to_wav(flac_path: str, wav_path: str) -> str:
#     """
#     Konversi file FLAC ke WAV menggunakan soundfile.
#     Mengembalikan path file WAV hasil konversi.
#     """
#     try:
#         data, samplerate = sf.read(flac_path)
#         sf.write(wav_path, data, samplerate, subtype="PCM_16")
#         print(f"[Konversi] {flac_path} -> {wav_path}")
#         print(f"  Sample rate : {samplerate} Hz")
#         print(f"  Jumlah sampel: {len(data)}")
#         return wav_path
#     except ImportError:
#         print('Error')


# ─────────────────────────────────────────────
# 1. LOAD AUDIO -> SAMPLING
# ─────────────────────────────────────────────
def load_and_sample(wav_path: str) -> tuple[list[float], int, int]:
    """
    Membaca file WAV dan mengambil nilai amplitudo kontinu (sampling).

    Returns:
        samples      : list nilai amplitudo float (kontinu, belum dikuantisasi)
        sample_rate  : jumlah sampel per detik (Hz)
        bit_depth    : kedalaman bit (misal 16)
    """
    with wave.open(wav_path, "rb") as wf:
        n_channels  = wf.getnchannels()
        sample_rate = wf.getframerate()
        bit_depth   = wf.getsampwidth() * 8          # bytes -> bits
        n_frames    = wf.getnframes()
        raw_bytes   = wf.readframes(n_frames)

    # Decode raw bytes -> integer PCM
    fmt = "<" + ("h" * (len(raw_bytes) // 2))        # little-endian int16
    pcm_int = list(struct.unpack(fmt, raw_bytes))

    # Ambil channel kiri saja jika stereo
    if n_channels == 2:
        pcm_int = pcm_int[::2]

    # Normalisasi ke rentang float [-1.0, +1.0]  (nilai kontinu hasil sampling)
    max_val = 2 ** (bit_depth - 1)                   # 32768 untuk 16-bit
    samples = [s / max_val for s in pcm_int]

    print(f"\n[Sampling]")
    print(f"  File        : {wav_path}")
    print(f"  Sample rate : {sample_rate} Hz")
    print(f"  Bit depth   : {bit_depth} bit")
    print(f"  Total sampel: {len(samples)}")
    print(f"  Durasi      : {len(samples)/sample_rate:.3f} detik")
    print(f"  Contoh 5 nilai sampling pertama:")
    for i, v in enumerate(samples[:5]):
        print(f"    samples[{i}] = {v:.10E}")

    return samples, sample_rate, bit_depth


# ─────────────────────────────────────────────
# 2. QUANTIZATION  (16-bit PCM)
# ─────────────────────────────────────────────
def quantize(samples: list[float], bit_depth: int = 16) -> list[float]:
    """
    Quantization 16-bit sesuai Bab 3.3.3.2:

    Langkah:
      1. Tentukan jumlah level  = 2^bit_depth  (65.536 untuk 16-bit)
         Rentang level           : -32768 s/d +32767
      2. Buat kode PCM           : kode = round(amplitudo_kontinu × 32767)
      3. Rekonstruksi ke float   : amplitudo_diskrit = kode / 32767

    Returns:
        quantized : list nilai amplitudo diskrit (sudah terkuantisasi)
    """
    n_levels  = 2 ** bit_depth                        # 65.536
    max_level = (n_levels // 2) - 1                   # 32767
    min_level = -(n_levels // 2)                      # -32768

    quantized = []
    for amp in samples:
        # Step 2 – buat kode PCM
        code = round(amp * max_level)
        code = max(min_level, min(max_level, code))   # clipping

        # Step 3 – rekonstruksi ke float
        amp_diskrit = code / max_level
        quantized.append(amp_diskrit)

    print(f"\n[Quantization]")
    print(f"  Bit depth   : {bit_depth} bit")
    print(f"  Jumlah level: {n_levels}  ({min_level} s/d +{max_level})")
    print(f"  Contoh perbandingan Sampling vs Quantization:")
    print(f"  {'Sampling':>20}  {'Quantization':>20}")
    print(f"  {'-'*44}")
    for i in range(min(20, len(samples))):
        print(f"  {samples[i]:>20.10E}  {quantized[i]:>20.10f}")

    return quantized


# ─────────────────────────────────────────────
# 3. HITUNG QUANTIZATION ERROR
# ─────────────────────────────────────────────
def quantization_error(
    samples: list[float], quantized: list[float]
) -> dict[str, float]:
    """
    Menghitung metrik error kuantisasi:
      - MSE  (Mean Squared Error)
      - SQNR (Signal-to-Quantization-Noise Ratio) dalam dB
    """
    n = len(samples)
    mse = sum((s - q) ** 2 for s, q in zip(samples, quantized)) / n

    signal_power = sum(s ** 2 for s in samples) / n
    sqnr_db = 10 * math.log10(signal_power / mse) if mse > 0 else float("inf")

    # Estimasi daya kuantisasi teoritis  (Δ²/12)
    max_level = (2 ** 16 // 2) - 1                    # 32767
    delta     = 1 / max_level                         # jarak antar level
    est_noise_power = (delta ** 2) / 12

    print(f"\n[Quantization Error]")
    print(f"  MSE (Mean Squared Error)   : {mse:.10E}")
    print(f"  Signal Power               : {signal_power:.10E}")
    print(f"  Est. Noise Power (Δ²/12)   : {est_noise_power:.10E}")
    print(f"  SQNR                       : {sqnr_db:.4f} dB")

    return {"mse": mse, "signal_power": signal_power, "sqnr_db": sqnr_db}


# ─────────────────────────────────────────────
# 4. DEMO TANPA FILE AUDIO  (data sintetis)
# ─────────────────────────────────────────────
def demo_synthetic():
    """
    Demo menggunakan data sintetis yang mencerminkan
    contoh tabel 3.3.3.2.1 pada dokumen.
    """
    print("=" * 60)
    print("DEMO: Sampling & Quantization (data sintetis)")
    print("=" * 60)

    # Nilai amplitudo kontinu (hasil sampling) – dari tabel dokumen
    raw_samples = [
        3.644620e-04,  4.356895e-04,  4.617208e-04,  4.618590e-04,
        3.899573e-04,  4.148397e-04,  5.153982e-04,  5.346136e-04,
        5.162396e-04,  4.093032e-04,  1.726370e-04,  1.794831e-05,
       -3.869251e-05, -1.529391e-04, -1.824075e-04, -1.856561e-05,
        1.033091e-04,  1.269404e-04,  3.065602e-04,  6.393479e-04,
    ]

    bit_depth = 16
    n_levels  = 2 ** bit_depth          # 65.536
    max_level = (n_levels // 2) - 1     # 32767
    min_level = -(n_levels // 2)        # -32768

    print(f"\n[Sampling]  – {len(raw_samples)} nilai amplitudo kontinu")
    print(f"\n[Quantization]")
    print(f"  Bit depth   : {bit_depth} bit")
    print(f"  Jumlah level: {n_levels}  ({min_level} s/d +{max_level})")
    print(f"\n  {'No':>3}  {'Sampling':>18}  {'Kode PCM':>10}  {'Quantization':>18}")
    print(f"  {'-'*55}")

    quantized = []
    for i, amp in enumerate(raw_samples):
        # Step 2 – kode PCM
        code = round(amp * max_level)
        code = max(min_level, min(max_level, code))
        # Step 3 – rekonstruksi
        amp_diskrit = code / max_level
        quantized.append(amp_diskrit)
        print(f"  {i:>3}  {amp:>18.10E}  {code:>10}  {amp_diskrit:>18.10f}")

    # Error
    mse = sum((s - q) ** 2 for s, q in zip(raw_samples, quantized)) / len(raw_samples)
    sig_pow = sum(s ** 2 for s in raw_samples) / len(raw_samples)
    sqnr = 10 * math.log10(sig_pow / mse) if mse > 0 else float("inf")

    print(f"\n[Quantization Error]")
    print(f"  MSE   : {mse:.6E}")
    print(f"  SQNR  : {sqnr:.4f} dB")
    print("\n" + "=" * 60)


# ─────────────────────────────────────────────
# 5. PIPELINE LENGKAP  (untuk file nyata)
# ─────────────────────────────────────────────
def run_pipeline(audio_path: str, bit_depth: int = 16):
    """
    Pipeline lengkap:
      FLAC/WAV -> Sampling -> Quantization -> Error report
    """
    print("=" * 60)
    print("PIPELINE: Sampling & Quantization")
    print("=" * 60)

    # Konversi FLAC jika perlu
    if audio_path.lower().endswith(".flac"):
        wav_path = audio_path.replace(".flac", "_converted.wav")
        audio_path = convert_flac_to_wav(audio_path, wav_path)

    # Sampling
    samples, sample_rate, detected_bit = load_and_sample(audio_path)
    bit_depth = detected_bit or bit_depth

    # Quantization
    quantized = quantize(samples, bit_depth)

    # Error
    quantization_error(samples, quantized)

    return quantized, sample_rate


# ─────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────
if __name__ == "__main__":
    import sys

    if len(sys.argv) > 1:
        # Jalankan pipeline dengan file yang diberikan
        audio_file = sys.argv[1]
        if not os.path.exists(audio_file):
            print(f"[Error] File tidak ditemukan: {audio_file}")
            sys.exit(1)
        run_pipeline(audio_file)
    else:
        # Jalankan demo sintetis (tidak butuh file audio)
        demo_synthetic()